In [0]:
# dbutils.fs.mkdirs("/Volumes/ska_catalog/bronze/bronze_volume/Emp_vs_Dep")

In [0]:
import pandas as pd
import numpy as np

df_department = pd.read_csv(r'/Volumes/ska_catalog/bronze/bronze_volume/Emp_vs_Dep/departments.csv',header = 0, delimiter=',')
df_employees = pd.read_csv(r'/Volumes/ska_catalog/bronze/bronze_volume/Emp_vs_Dep/employees.csv',header = 0, delimiter=',')

df_department.display()
df_employees.display()

In [0]:
var = list(df_department.columns)
print(type(var))

df_department = df_department.drop_duplicates(subset= var).reset_index(drop=True)
df_employees = df_employees.drop_duplicates(subset = list(df_employees.columns)).reset_index(drop = True)

df_department.display()
df_employees.display()


In [0]:
%sql
-- DROP TABLE ska_catalog.bronze.employee

In [0]:
%skip
spark_depart = spark.createDataFrame(df_department)
spark_emp = spark.createDataFrame(df_employees)

spark_depart.write.mode("overwrite").saveAsTable("ska_catalog.bronze.department")
# Available modes: 'append', 'overwrite', 'ignore', 'error' or 'errorifexists'
spark_emp.write.mode("ignore").saveAsTable("ska_catalog.bronze.employee")

In [0]:
df_department.head(5).display()
df_department.tail(5).display()
print(df_department.info())
var =df_department['DEPARTMENT_ID'].values
print(var)
df_department.shape
df_department.describe()


In [0]:
df_department.rename(lambda x: x.lower(),axis = 1)

In [0]:
df_department['MANAGER_ID'] = df_department['MANAGER_ID'].map(lambda x: None if x == ' - ' else x)
df_department.display()

In [0]:
today = pd.Timestamp.today().date()
sixmonthsbefore = pd.Timedelta('180 days')
print(sixmonthsbefore)
no_of_days = today - sixmonthsbefore
print(no_of_days)

In [0]:
%sql
SELECT * FROM ska_catalog.bronze.employee

In [0]:
df_emp_depart = pd.merge(df_employees, df_department, how = 'inner', on = 'DEPARTMENT_ID', sort = True)

df_emp_depart = df_emp_depart[['EMPLOYEE_ID','FIRST_NAME','SALARY','DEPARTMENT_NAME']]

In [0]:
df_emp_depart['rank'] = df_emp_depart.groupby("DEPARTMENT_NAME")['SALARY'].rank(method='dense', ascending=False)
top5 = df_emp_depart[df_emp_depart['rank'] <= 5]
top5.sort_values(by=['DEPARTMENT_NAME', 'rank']).display()

In [0]:
filtered_employees = df_employees[(df_employees['JOB_ID'].str.startswith('FI')) & (df_employees['SALARY'] > 8000)]
display(filtered_employees)

In [0]:
df_mean = pd.merge(df_employees, df_department, how= 'inner', on = ['DEPARTMENT_ID'], sort = True )
df_mean = df_mean\
    .groupby(["DEPARTMENT_ID", "DEPARTMENT_NAME"])\
    .agg({'SALARY': 'mean'})\
    .round(4)\
    .rename(columns={'SALARY': 'AVG SALARY'})\
    .sort_values(by='AVG SALARY', ascending=False)\
    .reset_index()
df_mean.display()

In [0]:
df_employees['MANAGER_ID'] = df_employees['MANAGER_ID'].map(lambda x : None if x == ' - ' else x)

df_employees['MANAGER_ID'] = pd.to_numeric(df_employees['MANAGER_ID'], errors='coerce').astype('Int64')
df_employees.display()

In [0]:
df_self_join = pd.merge(df_employees, df_employees, left_on = 'MANAGER_ID', right_on = 'EMPLOYEE_ID')

df_self_join['employee_name'] = df_self_join['FIRST_NAME_x'] + ' ' + df_self_join['LAST_NAME_x']
df_self_join['manager_name'] = df_self_join['FIRST_NAME_y'] + ' ' + df_self_join['LAST_NAME_y']
df_self_join = df_self_join[['employee_name', 'manager_name']]

df_count_per_manager = df_self_join\
    .groupby('manager_name').agg({'employee_name': 'count'})\
    .rename(columns={'employee_name': 'NO OF EMPLOYEES'})\
    .sort_values(by='NO OF EMPLOYEES', ascending=False)\
    .reset_index()
df_count_per_manager.display()

In [0]:
df_employees.display()
df_department.display()

In [0]:
df_employees['HIRE_DATE'] = pd.to_datetime(df_employees['HIRE_DATE'], format = '%d-%b-%y', errors = 'raise' )
print(df_employees['HIRE_DATE'])

In [0]:
df_employees['HIRE_DATE'] = pd.to_datetime(df_employees['HIRE_DATE'], format = '%d-%b-%y', errors = 'raise' )

df_employees[ df_employees['HIRE_DATE'].dt.year == 2005].display()

In [0]:
df_employee_cnt = df_employees.groupby(df_employees['HIRE_DATE'].dt.year)\
    .agg({'EMPLOYEE_ID': 'count'})\
    .reset_index()\
    .rename(columns = {"EMPLOYEE_ID" : 'EMP_COUNT', 'HIRE_DATE': 'YEAR'})\
    .sort_values(by = ['YEAR','EMP_COUNT'], ascending = [True, True])

df_employee_cnt.display()